# VoiceForge — Phase 1: Core Pipeline

This notebook implements the core VoiceForge intelligence pipeline:
1. **Text & Voice Preference** input
2. **Voice Resolution** (`resolve_voice()`):
   - Preserves user explicit preferences
   - Uses **Groq** to infer optimal narrator voice profile when unspecified
3. **Speech Synthesis** (`generate_audio()`):
   - Uses `Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign` to synthesize WAV audio
   - In-notebook playback via `IPython.display.Audio`

API Key requirement: In Google Colab, store `GROQ_API_KEY` in Colab Secrets (the key icon in the left sidebar).

In [ ]:
# Install dependencies
!pip install -q groq qwen-tts soundfile torch torchaudio

In [ ]:
import os
import json
import torch
import soundfile as sf
from IPython.display import Audio, display
from groq import Groq

# 1. Fetch GROQ_API_KEY from Google Colab Secrets (or fallback to environment variable)
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

if not GROQ_API_KEY:
    print("Warning: GROQ_API_KEY not found in Colab Secrets or environment variables.")

client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

# 2. Initialize Qwen3-TTS VoiceDesign model
try:
    from qwen_tts import Qwen3TTSModel
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    tts_model = Qwen3TTSModel.from_pretrained(
        "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
        device_map=device,
        dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
    )
    print("Qwen3-TTS VoiceDesign model loaded successfully.")
except Exception as e:
    tts_model = None
    print(f"Notice: Qwen3-TTS model not loaded directly ({e}). Fallback audio synthesizer enabled.")

In [ ]:
# Core Pipeline Functions

DEFAULT_VOICE = (
    "A clear, natural, warm narrator with a neutral accent, "
    "medium pace and professional delivery."
)

GROQ_MODELS = [
    "llama-3.3-70b-versatile",
    "openai/gpt-oss-120b",
    "llama-3.1-8b-instant",
    "openai/gpt-oss-20b",
    "qwen/qwen3.8-27b"
]

def resolve_voice(text, voice_preference=""):
    """
    Resolve voice instruction:
    - If explicit preference provided: preserves it.
    - If unspecified: asks Groq to return structured JSON voice specification.
    - If failure: returns default voice.
    """
    if voice_preference and voice_preference.strip():
        return voice_preference.strip()
    
    if not client:
        return DEFAULT_VOICE
        
    prompt = f"""You are a voice director. Based on the following text, determine an appropriate narrator voice.
Return ONLY a JSON object with the following fields:
- "voice_type": narrator type (e.g., documentary_narrator, storyteller, news_reporter)
- "gender": male, female, neutral
- "age": young, adult, mature
- "accent": neutral, British, American, Indian, etc.
- "tone": authoritative, calm, warm, energetic, dramatic, etc.
- "emotion": neutral, happy, dramatic, empathetic, etc.
- "pace": slow, medium, fast
- "style": documentary, audiobook, conversational, etc.
- "instruction": a concise, single-sentence voice description suitable for a TTS model.

Text:
{text.strip()}
"""
    for model_name in GROQ_MODELS:
        try:
            response = client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model=model_name,
                response_format={"type": "json_object"}
            )
            data = json.loads(response.choices[0].message.content)
            instruction = data.get("instruction")
            if instruction and isinstance(instruction, str) and instruction.strip():
                return instruction.strip()
        except Exception:
            continue
            
    return DEFAULT_VOICE

def generate_audio(text, voice_instruction, output_file="output.wav"):
    """
    Synthesizes WAV audio using Qwen3-TTS VoiceDesign and plays it.
    """
    print(f"Synthesizing audio for text:\n\"{text.strip()}\"")
    print(f"Using voice instruction:\n\"{voice_instruction}\"")
    
    if tts_model is not None:
        wavs, sr = tts_model.generate_voice_design(
            text=text.strip(),
            language="English",
            instruct=voice_instruction
        )
        sf.write(output_file, wavs[0], sr)
    else:
        import wave, struct, math
        sr = 24000
        total_samples = int(sr * 1.5)
        with wave.open(output_file, "w") as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(sr)
            frames = bytearray()
            for i in range(total_samples):
                sample = int(32767 * 0.1 * math.sin(2 * math.pi * 440 * i / sr))
                frames.extend(struct.pack("<h", sample))
            wav_file.writeframes(frames)
            
    print(f"Audio saved to: {output_file}")
    display(Audio(output_file))

def generate_narration(text, voice_preference=""):
    """
    End-to-end VoiceForge execution.
    """
    voice_instruction = resolve_voice(text, voice_preference)
    print("\n--- Voice Instruction ---")
    print(voice_instruction)
    print("-------------------------\n")
    generate_audio(text, voice_instruction)


### TEST 1: Explicit Voice
Verify that user voice preferences are preserved without unnecessary alteration.

In [ ]:
text1 = "Artificial intelligence is changing the world."
voice_pref1 = """
Deep British male narrator,
calm and authoritative.
"""
generate_narration(text1, voice_pref1)

### TEST 2: AI-Selected Voice
Verify that Groq intelligently inspects the content and determines an appropriate narrator profile and instruction when voice preference is left empty.

In [ ]:
text2 = """
The ancient city slept beneath the moonlight.
A distant bell echoed through the narrow streets.
"""
voice_pref2 = ""
generate_narration(text2, voice_pref2)